# Lab 5B — Forecasting with causal and dilated CNNs
### Engineering MSc · Lecture 5, slides 99–262 · 120-minute core + optional extensions

**Engineering question:** Can a network predict a sensor response when the relevant input happened 32 samples ago?

Lab 5A introduced local filters and pooling for window classification. Here we investigate **which observations a network can use**, then design a direct ten-step forecaster.

You will:
- calculate and visualize receptive fields, including gaps caused by dilation;
- test causality by changing observations after a chosen time;
- compare equally sized residual CNNs with different dilation schedules;
- compare final-position selection with global average pooling;
- match prediction-head shapes to the task;
- inspect WaveNet-style gating and parallel multiscale branches in optional extensions.

All code is supplied and commented. **Run the cells in order.** Predict and explain the results before changing the clearly marked settings. No external dataset or GPU is required. Sections 1–5 run with NumPy, Matplotlib, and scikit-learn. Training also requires Keras 3 and PyTorch, as in Lab 5A.

**Main deliverable:** your executed notebook and a short explanation of which information each architecture could access. A larger model is not automatically a better model.


## 0. Setup — 5 minutes

Use a fresh Python kernel. Install the packages listed in `requirements.txt` if needed. In a notebook, `%pip install -r requirements.txt` installs into the current kernel's environment; restart the kernel afterward.

The line setting `KERAS_BACKEND` must run **before importing Keras**. Do not change the backend in a kernel that has already imported Keras.


In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

# Match Lab 5A: Keras API with a PyTorch backend; a CPU is sufficient.
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

SEED = 21
np.random.seed(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})

# Main settings. Keep these fixed for the first comparison.
L = 64                   # historical samples supplied to the network
H = 10                   # future values predicted in one forward pass
DELAY = 32               # delay in the synthetic data-generating system
FS = 20                  # nominal sample rate, in samples per second
WIDTH = 16               # channels inside each residual block
EPOCHS = 18               # maximum epochs per neural model
BATCH_SIZE = 64
RUN_EXTRA_TRAINING = False  # optional WaveNet/multiscale training, not core

print('Python:', sys.version.split()[0])
print(f'History: {L} samples | Forecast: {H} samples | Delay: {DELAY} samples')


## 1. How much history reaches one output? — 15 minutes

For a stack of temporal layers, track two quantities:

$$R_0=1,\quad J_0=1$$
$$R_\ell=R_{\ell-1}+(K_\ell-1)D_\ell J_{\ell-1},\qquad
J_\ell=J_{\ell-1}S_\ell.$$

- $R_\ell$: span of the receptive field in original input samples.
- $J_\ell$: spacing between adjacent outputs, measured in original input samples.
- $K_\ell,D_\ell,S_\ell$: kernel size, dilation, and stride of layer $\ell$.

Pooling layers can be included using their window size and stride. A pointwise convolution has $K=1$ and does not expand the temporal span. For a residual block, follow the longest dependency path.

For stride-one convolutions, $R=1+\sum_\ell(K_\ell-1)D_\ell$. **Count each convolution, not each block.** Our TCN has **two serial convolutions per block**, so for $K=3$:

$$R=1+2(3-1)\sum_b D_b.$$

**Predict:** What are the receptive fields for four blocks with dilations `[1,1,1,1]` and `[1,2,4,8]`? Does changing dilation change the number of kernel weights?


In [ ]:
def receptive_field(layers):
    """layers contains (kernel size, dilation, stride) for each serial layer."""
    span, jump = 1, 1
    for kernel, dilation, stride in layers:
        span += (kernel - 1) * dilation * jump
        jump *= stride
    return span, jump

def block_layers(dilations, kernel=3):
    # Each residual block contains TWO convolutions at the same dilation.
    return [(kernel, d, 1) for d in dilations for _ in range(2)]

SHORT_D = [1, 1, 1, 1]
LONG_D = [1, 2, 4, 8]
R_SHORT, _ = receptive_field(block_layers(SHORT_D))
R_LONG, _ = receptive_field(block_layers(LONG_D))

print('Short residual CNN: RF =', R_SHORT)
print('Dilated residual CNN: RF =', R_LONG)
print('Conv K3 -> Pool K2/S2 -> Conv K3:',
      receptive_field([(3,1,1), (2,1,2), (3,1,1)]))
print(f'Last-output temporal extent: {(R_LONG-1)/FS:.2f} seconds between oldest and newest samples')
assert R_SHORT == 17 and R_LONG == 61
assert receptive_field([(3,1,1), (2,1,2), (3,1,1)]) == (8,2)


### A wide span can still contain holes

The receptive-field span gives the oldest and newest accessible positions. It does not guarantee that **every** position between them is connected. The next calculation lists all structurally reachable lags for a stride-one stack. Lag zero is the current sample.

**Try it:** Compare the regular dilation schedule with `[4,4,4,4]`. Can a feature at lag 1 influence the latter stack?


In [ ]:
def reachable_lags(kernel, dilations):
    """Exact dependency offsets for serial stride-one convolutions."""
    lags = {0}
    for d in dilations:
        # Every new output combines K positions in the previous feature map.
        lags = {old + tap*d for old in lags for tap in range(kernel)}
    return np.array(sorted(lags))

schedules = {
    'Eight ordinary convolutions': [1]*8,
    'Four two-convolution dilated blocks': [1,1,2,2,4,4,8,8],
    'Four convolutions, all dilation 4': [4]*4,
}
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
for ax, (name, dilations) in zip(axes, schedules.items()):
    lags = reachable_lags(3, dilations)
    ax.scatter(lags, np.zeros(len(lags)), marker='s', s=30)
    ax.set_yticks([])
    ax.set_title(f'{name}: span {lags[-1]+1}, accessible positions {len(lags)}', fontsize=10)
axes[-1].set_xlabel('Lag relative to the output time (samples into the past)')
plt.tight_layout()
plt.show()
assert np.array_equal(reachable_lags(3, [1,1,2,2,4,4,8,8]), np.arange(61))


## 2. A direct causality test — 15 minutes

For an online output at time $t$, changing inputs **after $t$** must not change that output. We first test fixed filters so the result does not depend on training.

The function below reproduces a single-channel, stride-one deep-learning cross-correlation with an odd kernel size. Causal padding adds $(K-1)D$ zeros on the left. Symmetric `same` padding places zeros on both sides.

**Predict:** After replacing the suffix following time 30, which outputs can change for each padding mode?


In [ ]:
def conv1d_numpy(signal, weights, dilation=1, padding='causal'):
    """Transparent one-channel convolution for this diagnostic only."""
    effective_span = 1 + (len(weights)-1)*dilation
    if padding == 'causal':
        left, right = effective_span-1, 0
    elif padding == 'same':
        left = (effective_span-1)//2
        right = effective_span-1-left
    else:
        raise ValueError('Choose causal or same')
    padded = np.pad(signal, (left, right))
    # The loop exposes exactly which samples contribute to each output.
    return np.array([np.dot(padded[t:t+effective_span:dilation], weights)
                     for t in range(len(signal))])

rng = np.random.default_rng(4)
signal = rng.normal(size=64)
cut = 30
changed = signal.copy()
changed[cut+1:] += 4.0  # keep the prefix through cut identical
weights = np.array([0.2, 0.3, 0.5])

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, padding in zip(axes, ['same', 'causal']):
    before = conv1d_numpy(signal, weights, padding=padding)
    after = conv1d_numpy(changed, weights, padding=padding)
    difference = np.abs(after-before)
    prefix_change = difference[:cut+1].max()
    print(padding, '| maximum output change at or before cut:', prefix_change)
    ax.plot(difference)
    ax.axvline(cut, color='black', linestyle='--', label='last unchanged input')
    ax.set_title(f'{padding} padding')
    ax.set_ylabel('Output change')
    ax.legend()
    if padding == 'causal': assert np.allclose(difference[:cut+1], 0)
    else: assert prefix_change > 0
axes[-1].set_xlabel('Output time index')
plt.tight_layout()
plt.show()


## 3. Global pooling or the final representation? — 10 minutes

A temporal encoder produces $H\in\mathbb{R}^{B\times T\times C}$.

| Readout | Operation | Output shape | Information directly collected |
|---|---|---|---|
| Global average | $\bar h=\frac{1}{T}\sum_t h_t$ | $(B,C)$ | Every temporal position |
| Final position | $h=h_T$ | $(B,C)$ | The final encoded position |
| Dense at every position | $\hat y_t=Wh_t+b$ | $(B,T,Q)$ | Separate output vector at each position |

The final encoded vector can contain earlier information **within its receptive field**. It is not the last raw measurement. Global pooling covers the whole window but does not itself learn the ordering of distant events. Order may already be encoded in the input feature channels.

**Predict:** If two feature maps contain the same activation at different times, what happens to their global mean and their final value? These are deliberately constructed feature maps, not a claim that arbitrary CNN encoders produce identical averages for shifted signals.


In [ ]:
# Two already-encoded feature maps: [batch=2, time=8, channels=1].
features = np.zeros((2, 8, 1))
features[0, 1, 0] = 4.0   # event early in the window
features[1, 7, 0] = 4.0   # event at the end
pooled = features.mean(axis=1)
last = features[:, -1, :]
print('Global averages:', pooled.ravel())
print('Final values:   ', last.ravel())

# Dense operates on the last axis; it does not collapse the time axis.
W = np.ones((1, H))
print('Dense(H) at every position:', (features @ W).shape)
print('Dense(H) after global mean:', (pooled @ W).shape)
print('Dense(H) after final selection:', (last @ W).shape)
assert pooled.shape == last.shape == (2,1)


## 4. A system with a known delay — 15 minutes

Our synthetic system has an observed input command $u_t$ and measured response $y_t$:

$$y_t=0.5y_{t-1}+0.8u_{t-32}+\varepsilon_t,\qquad
u_t\sim\mathcal N(0,1),\quad\varepsilon_t\sim\mathcal N(0,0.04^2).$$

This is a teaching model with a delayed input and first-order dynamics, not a validated model of a particular device. Both $u$ and $y$ are available in the historical window. Future commands are **not** supplied.

At origin $t$, the input is `[u,y]` for $t-63,\ldots,t$. The target is $[y_{t+1},\ldots,y_{t+10}]$.

For horizon $h$, the directly required command is $u_{t+h-32}$. Its lag relative to the origin is $32-h$, ranging from **31 to 22**. All are in the 64-sample input window, but all are outside a final representation with receptive field 17. Earlier response measurements cannot reveal these independent commands because the delayed effect has not yet arrived.

We generate **separate runs for training, validation, and testing, before constructing windows**. Windows may overlap within a run, but no run appears in two partitions. Scaling is fitted on the training runs only.


In [ ]:
def generate_runs(count, seed, length=256, delay=DELAY):
    """Independent synthetic experiments, each with two recorded channels."""
    rng = np.random.default_rng(seed)
    burn = 4*delay  # discard the startup period before recording the run
    total = length + burn
    u = rng.normal(size=(count, total))
    y = np.zeros_like(u)
    for t in range(1, total):
        delayed = u[:, t-delay] if t >= delay else np.zeros(count)
        y[:, t] = 0.5*y[:, t-1] + 0.8*delayed + rng.normal(0, 0.04, count)
    return np.stack([u[:, burn:], y[:, burn:]], axis=-1).astype('float32')

def make_windows(runs, split_name, length=L, horizon=H, step=8):
    """Build history/target pairs entirely inside each independent run."""
    X, Y, run_ids, origins = [], [], [], []
    for run_id, run in enumerate(runs):
        # The last target must still be inside this run.
        for t in range(length-1, len(run)-horizon, step):
            X.append(run[t-length+1:t+1])
            Y.append(run[t+1:t+1+horizon, 1])
            run_ids.append(f'{split_name}_{run_id}')
            origins.append(t)
    return np.asarray(X), np.asarray(Y), np.array(run_ids), np.array(origins)

train_runs = generate_runs(64, 101)
val_runs = generate_runs(16, 202)
test_runs = generate_runs(16, 303)
X_train_raw, Y_train_raw, train_ids, train_origins = make_windows(train_runs, 'train')
X_val_raw, Y_val_raw, val_ids, val_origins = make_windows(val_runs, 'val')
X_test_raw, Y_test_raw, test_ids, test_origins = make_windows(test_runs, 'test')

# One mean/std for each channel, calculated from TRAINING runs only.
mean = train_runs.mean(axis=(0,1), keepdims=True)
std = train_runs.std(axis=(0,1), keepdims=True)
std = np.maximum(std, 1e-6)
y_mean, y_std = float(mean[0,0,1]), float(std[0,0,1])

X_train = ((X_train_raw-mean)/std).astype('float32')
X_val = ((X_val_raw-mean)/std).astype('float32')
X_test = ((X_test_raw-mean)/std).astype('float32')
Y_train = ((Y_train_raw-y_mean)/y_std).astype('float32')
Y_val = ((Y_val_raw-y_mean)/y_std).astype('float32')

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)
assert np.allclose(X_train_raw[0], train_runs[0, :L])
assert np.allclose(Y_train_raw[0], train_runs[0, L:L+H, 1])
assert H <= DELAY, 'The known-system reference below assumes this condition.'
print('Training:', X_train.shape, Y_train.shape)
print('Validation:', X_val.shape, Y_val.shape)
print('Test:', X_test.shape, Y_test_raw.shape)
print('Training channel mean:', mean.ravel(), '| std:', std.ravel())


In [ ]:
# Inspect a TRAINING example. The target begins after the history ends.
i = 5
history_time = np.arange(-L+1, 1)/FS
future_time = np.arange(1, H+1)/FS
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(history_time, X_train_raw[i,:,0], label='observed command u')
needed_indices = (L-1) + np.arange(1,H+1) - DELAY
axes[0].scatter(history_time[needed_indices], X_train_raw[i,needed_indices,0],
                color='darkorange', label='commands affecting next ten responses')
axes[1].plot(history_time, X_train_raw[i,:,1], label='observed response y')
axes[1].plot(future_time, Y_train_raw[i], 'o-', color='darkorange', label='future target')
for ax in axes:
    ax.axvline(0, color='black', linestyle='--')
    ax.set_ylabel('Amplitude (a.u.)')
    ax.legend(fontsize=8)
axes[-1].set_xlabel('Time relative to forecast origin (seconds)')
plt.tight_layout()
plt.show()


## 5. Baselines before deep learning — 10 minutes

We use two deployable data-based baselines:

1. **Persistence:** repeat the most recent observed response at every horizon.
2. **Ridge regression:** learn a linear mapping from the flattened historical window to all ten future values.

The synthetic generator is linear, so ridge regression is a strong and appropriate benchmark. A neural network does not have to beat it for the receptive-field experiment to be informative.

We also calculate a **known-system reference** using the exact generator coefficients and zero expected future noise. It is an analytical sanity check, not a model learned from the training data. All commands it needs are already in the historical window because $H\leq32$.

Only validation results are inspected while making choices. The held-out test set is evaluated in Section 9 after the experiment is fixed.


In [ ]:
def persistence(X_raw):
    return np.repeat(X_raw[:, -1, 1:2], H, axis=1)

def known_system_reference(X_raw):
    """Conditional mean under the KNOWN teaching generator."""
    previous = X_raw[:, -1, 1].copy()
    result = []
    for h in range(1, H+1):
        # Index L-1 is the origin; index L-1+h-DELAY is an observed command.
        u_index = L-1+h-DELAY
        assert 0 <= u_index < L
        previous = 0.5*previous + 0.8*X_raw[:, u_index, 0]
        result.append(previous.copy())
    return np.stack(result, axis=1)

def rmse_by_horizon(truth, prediction):
    return np.sqrt(np.mean((truth-prediction)**2, axis=0))

def overall_rmse(truth, prediction):
    return float(np.sqrt(np.mean((truth-prediction)**2)))

ridge = Ridge(alpha=1.0)  # fixed before looking at validation/test performance
ridge.fit(X_train.reshape(len(X_train), -1), Y_train)
val_predictions = {
    'Persistence': persistence(X_val_raw),
    'Ridge': ridge.predict(X_val.reshape(len(X_val), -1))*y_std+y_mean,
    'Known-system reference': known_system_reference(X_val_raw),
}
for name, pred in val_predictions.items():
    print(f'{name:26s} validation RMSE: {overall_rmse(Y_val_raw,pred):.4f}')

fig, ax = plt.subplots(figsize=(8,3))
for name, pred in val_predictions.items():
    ax.plot(np.arange(1,H+1), rmse_by_horizon(Y_val_raw,pred), marker='o', label=name)
ax.set(xlabel='Forecast horizon (samples)', ylabel='Validation RMSE (a.u.)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 6. Three neural models with matched parameter counts — 15 minutes

We now import Keras. The API is the same as Lab 5A, with `KERAS_BACKEND='torch'`. Every residual block contains two causal convolutions. We use no temporal downsampling and no normalization that mixes time positions.

| Model | Block dilations | Final-position receptive field | Readout |
|---|---|---:|---|
| Short / last | 1, 1, 1, 1 | 17 | Final position |
| Dilated / last | 1, 2, 4, 8 | 61 | Final position |
| Dilated / mean | 1, 2, 4, 8 | 61 per position | Global average |

The pooled model's prediction combines all positions, so **61 is not a restriction on the total window information available to its final prediction**. It is the span of each feature vector before pooling.

All models have the same channel width, number of convolutional layers, dense head size, initialization seed, optimizer, batch size, maximum epochs, and early-stopping rule. Dilation and readout introduce no extra trainable weights. This is an architectural comparison on one controlled task, not a general model ranking.


In [ ]:
import keras
import torch

if keras.backend.backend() != 'torch':
    raise RuntimeError('Restart the kernel and run Section 0 before importing Keras.')
torch.set_num_threads(2)
keras.config.set_floatx('float32')
keras.utils.set_random_seed(SEED)
print('Keras', keras.__version__, '| PyTorch', torch.__version__)

def as_numpy(value):
    # Safe for tensors returned by any Keras backend, including PyTorch tensors.
    return keras.ops.convert_to_numpy(value)

def residual_block(x, width, dilation, name):
    shortcut = x  # short route that will be added back to the transformed input
    z = keras.layers.Conv1D(width, 3, padding='causal', dilation_rate=dilation,
                            activation='relu', name=name+'_conv1')(x)
    z = keras.layers.Conv1D(width, 3, padding='causal', dilation_rate=dilation,
                            name=name+'_conv2')(z)
    # Here input/output widths match, so an identity shortcut is sufficient.
    z = keras.layers.Add(name=name+'_add')([shortcut, z])
    return keras.layers.Activation('relu', name=name+'_relu')(z)

def build_tcn(dilations, readout='last', name='tcn'):
    inputs = keras.Input(shape=(L,2), name=name+'_history')
    # K=1 mixes the two channels without extending temporal context.
    x = keras.layers.Conv1D(WIDTH, 1, name=name+'_project')(inputs)
    for block, dilation in enumerate(dilations):
        x = residual_block(x, WIDTH, dilation, name=f'{name}_block{block}')
    backbone = keras.Model(inputs, x, name=name+'_backbone')
    if readout == 'last':
        # All samples in the batch, last time index, all feature channels.
        h = keras.layers.Lambda(lambda z: z[:, -1, :], name=name+'_last')(x)
    elif readout == 'mean':
        h = keras.layers.GlobalAveragePooling1D(name=name+'_mean')(x)
    else:
        raise ValueError('readout must be last or mean')
    outputs = keras.layers.Dense(H, name=name+'_forecast')(h)  # linear regression outputs
    model = keras.Model(inputs, outputs, name=name)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.002), loss='mse')
    return model, backbone

specs = {
    'Short / last': (SHORT_D, 'last', 'short_last'),
    'Dilated / last': (LONG_D, 'last', 'dilated_last'),
    'Dilated / mean': (LONG_D, 'mean', 'dilated_mean'),
}
models, backbones, extras = {}, {}, {}  # extras stays empty if the optional section is skipped
initial_weights = None
for label, (dilations, readout, name) in specs.items():
    keras.utils.set_random_seed(SEED)  # identical-shaped kernels start identically
    models[label], backbones[label] = build_tcn(dilations, readout, name)
    current_weights = models[label].get_weights()
    if initial_weights is None:
        initial_weights = current_weights
    else:
        assert all(np.allclose(a,b) for a,b in zip(initial_weights,current_weights))
    print(label, '| parameters:', models[label].count_params(),
          '| output:', models[label].output_shape)
assert len({m.count_params() for m in models.values()}) == 1
assert models['Dilated / last'].count_params() == 6490
models['Dilated / last'].summary()


### Check the actual Keras computations before fitting

We compare a fixed Keras convolution with the transparent NumPy calculation, then apply the suffix-change test to all three backbones. The tests concern **per-time-step features**. Pooling a full sequence into one vector is interpreted at the end of that full sequence.


In [ ]:
for padding in ['causal', 'same']:
    layer = keras.layers.Conv1D(1, 3, padding=padding, use_bias=False)
    _ = layer(signal[None,:,None].astype('float32'))  # build the layer
    layer.set_weights([weights.astype('float32').reshape(3,1,1)])
    observed = as_numpy(layer(signal[None,:,None].astype('float32'))).ravel()
    expected = conv1d_numpy(signal, weights, padding=padding)
    np.testing.assert_allclose(observed, expected, atol=1e-6)

def check_causal_backbone(backbone, sample, cut=30):
    changed = sample.copy()
    changed[:, cut+1:, :] += 3.0
    a = as_numpy(backbone(sample, training=False))
    b = as_numpy(backbone(changed, training=False))
    change = float(np.max(np.abs(a[:,:cut+1]-b[:,:cut+1])))
    assert change < 1e-5
    return change

for name, backbone in backbones.items():
    print(name, '| prefix change:', check_causal_backbone(backbone, X_val[:4]))

# A perturbation outside the short model's final receptive field cannot affect it.
probe = X_val[:4].copy()
probe[:, L-1-31, 0] += 5.0  # lag 31, relevant to the first forecast horizon
short_before = as_numpy(backbones['Short / last'](X_val[:4], training=False))[:, -1]
short_after = as_numpy(backbones['Short / last'](probe, training=False))[:, -1]
np.testing.assert_allclose(short_before, short_after, atol=1e-6)
print('Lag-31 perturbation cannot reach the short model final representation.')


## 7. Training and validation comparison — 10 minutes

**Before running:** Predict the expected difference between the two final-position models. Then predict what averaging might change. Write these as hypotheses, not guaranteed rankings.

Training uses only training examples. Validation loss chooses the best epoch. The same maximum budget and stopping rule apply to each model; realized epoch counts may differ. Hardware and software versions affect runtime and numerical results.


In [ ]:
histories, train_seconds, completed_epochs = {}, {}, {}

def fit_forecaster(model):
    # Reset the seed used for minibatch shuffling before each training run.
    keras.utils.set_random_seed(SEED+1)
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                         restore_best_weights=True)
    start = time.perf_counter()
    hist = model.fit(X_train, Y_train, validation_data=(X_val,Y_val),
                     epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
                     callbacks=[stop], verbose=0)
    return hist, time.perf_counter()-start

for name, model in models.items():
    histories[name], train_seconds[name] = fit_forecaster(model)
    completed_epochs[name] = len(histories[name].history['loss'])
    val_predictions[name] = model.predict(X_val, verbose=0)*y_std+y_mean
    print(f'{name:20s} | val RMSE {overall_rmse(Y_val_raw,val_predictions[name]):.4f}'
          f' | epochs {completed_epochs[name]} | seconds {train_seconds[name]:.1f}')

fig, axes = plt.subplots(1,2, figsize=(11,3.6))
for name, hist in histories.items():
    axes[0].plot(hist.history['val_loss'], label=name)
for name, pred in val_predictions.items():
    axes[1].plot(np.arange(1,H+1), rmse_by_horizon(Y_val_raw,pred), label=name)
axes[0].set(xlabel='Epoch index', ylabel='Validation MSE (scaled response)')
axes[1].set(xlabel='Forecast horizon (samples)', ylabel='Validation RMSE (a.u.)')
for ax in axes: ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 8. The same backbone, different output shapes — 5 minutes

**Predict each shape before running.** A dense layer acts on the last axis. It does not decide whether the task is classification, regression, or forecasting. That depends on targets, activation, loss, and the preceding readout.

For `[B,L,C]`, `Dense(10)` gives `[B,L,10]`: ten outputs **at each** historical position. For `[B,C]`, it gives `[B,10]`: one ten-dimensional output per window.


In [ ]:
backbone = backbones['Dilated / last']
encoded = backbone(X_val[:3], training=False)  # [3,L,WIDTH]
last_encoded = encoded[:, -1, :]              # [3,WIDTH]
head1 = keras.layers.Dense(1)
head10 = keras.layers.Dense(H)
print('Backbone features:             ', tuple(encoded.shape))
print('One forecast per window:       ', tuple(head1(last_encoded).shape))
print('Ten forecasts per window:      ', tuple(head10(last_encoded).shape))
print('Ten outputs at each time step: ', tuple(head10(encoded).shape))
assert tuple(head10(encoded).shape) == (3,L,H)
print('These new heads are untrained; this is a shape demonstration only.')


## Optional extension A. WaveNet-style gates and multiscale branches — 20–35 minutes

**WaveNet-style model.** At each block, parallel causal convolutions produce $a$ and $b$. The gate computes $z=\tanh(a)\odot\sigma(b)$. A residual path passes information to the next block; skip outputs are summed near the prediction head. Parallel gate branches have the same temporal span, so their contributions are not added when calculating the longest receptive-field path. With $K=2$ and dilations 1,2,4,8,16,32, the span is 64. This is an adaptation for direct forecasting, not a reproduction of the original audio-generation model.

**Multiscale model.** Parallel kernels of sizes 3, 9, and 33 operate on the same input. Concatenation stacks their channels. Its longest receptive-field span is 33, not $3+9+33$. This is an illustrative multiscale block, not the full InceptionTime architecture.

Build and inspect both models. Optional training is enabled by `RUN_EXTRA_TRAINING=True` in Section 0. These models have different parameter counts: interpret their results as an exploration, not a controlled test of one architectural change.


In [ ]:
def build_wavenet_style():
    inputs = keras.Input((L,2))
    x = keras.layers.Conv1D(WIDTH, 1)(inputs)
    skips = []
    for d in [1,2,4,8,16,32]:
        a = keras.layers.Conv1D(WIDTH, 2, padding='causal', dilation_rate=d,
                                activation='tanh')(x)
        b = keras.layers.Conv1D(WIDTH, 2, padding='causal', dilation_rate=d,
                                activation='sigmoid')(x)
        gated = keras.layers.Multiply()([a,b])  # channel-wise learned gate
        skips.append(keras.layers.Conv1D(WIDTH, 1)(gated))
        residual = keras.layers.Conv1D(WIDTH, 1)(gated)
        x = keras.layers.Add()([x,residual])
    x = keras.layers.Add()(skips)  # combine skip contributions across blocks
    x = keras.layers.Activation('relu')(x)
    backbone = keras.Model(inputs, x)
    h = keras.layers.Lambda(lambda z: z[:, -1, :])(x)
    model = keras.Model(inputs, keras.layers.Dense(H)(h), name='wavenet_style')
    model.compile(optimizer=keras.optimizers.Adam(0.002), loss='mse')
    return model, backbone

def build_multiscale():
    inputs = keras.Input((L,2))
    branches = [keras.layers.Conv1D(8, k, padding='causal', activation='relu')(inputs)
                for k in [3,9,33]]
    # Concatenate CHANNELS, preserving the common batch and time dimensions.
    x = keras.layers.Concatenate(axis=-1)(branches)  # [B,L,24]
    x = keras.layers.Conv1D(WIDTH, 1, activation='relu')(x)
    backbone = keras.Model(inputs, x)
    h = keras.layers.Lambda(lambda z: z[:, -1, :])(x)
    model = keras.Model(inputs, keras.layers.Dense(H)(h), name='multiscale')
    model.compile(optimizer=keras.optimizers.Adam(0.002), loss='mse')
    return model, backbone

extras = {}
for name, builder, rf in [('WaveNet-style', build_wavenet_style, 64),
                          ('Multiscale', build_multiscale, 33)]:
    keras.utils.set_random_seed(SEED)
    model, backbone = builder()
    assert tuple(model(X_val[:3]).shape) == (3,H)
    print(name, '| parameters:', model.count_params(), '| RF:', rf,
          '| prefix change:', check_causal_backbone(backbone, X_val[:4]))
    if RUN_EXTRA_TRAINING:
        histories[name], train_seconds[name] = fit_forecaster(model)
        completed_epochs[name] = len(histories[name].history['loss'])
        extras[name] = model
        val_predictions[name] = model.predict(X_val, verbose=0)*y_std+y_mean
        print('Validation RMSE:', overall_rmse(Y_val_raw,val_predictions[name]))


## 9. Held-out evaluation — 10 minutes

Freeze your choices before this section. Evaluate all predeclared comparisons on the test partition, including baselines, and report error by horizon. Do not return to tune the models on these test results.

Overlapping windows from the same run are correlated. We therefore also report the variation of RMSE **across independent test runs**. The standard deviation across runs is descriptive variation, not a confidence interval for a model ranking.

**Try it:** Which model would you choose for this particular task once error, simplicity, and parameter count are considered? Does the answer support the claim that deep learning is always necessary?


In [ ]:
test_predictions = {
    'Persistence': persistence(X_test_raw),
    'Ridge': ridge.predict(X_test.reshape(len(X_test), -1))*y_std+y_mean,
    'Known-system reference': known_system_reference(X_test_raw),
}
# Fixed baseline results can be inspected even without neural dependencies.
for name, pred in test_predictions.items():
    print(f'{name:26s} test RMSE: {overall_rmse(Y_test_raw,pred):.4f}')


In [ ]:
for name, model in {**models, **extras}.items():
    test_predictions[name] = model.predict(X_test, verbose=0)*y_std+y_mean

for name, pred in test_predictions.items():
    per_run = np.array([overall_rmse(Y_test_raw[test_ids==r], pred[test_ids==r])
                        for r in np.unique(test_ids)])
    print(f'{name:26s} pooled RMSE={overall_rmse(Y_test_raw,pred):.4f}'
          f' | run mean={per_run.mean():.4f} | run SD={per_run.std(ddof=1):.4f}')

fig, ax = plt.subplots(figsize=(10,4))
for name, pred in test_predictions.items():
    ax.plot(np.arange(1,H+1), rmse_by_horizon(Y_test_raw,pred), marker='o', label=name)
ax.set(xlabel='Forecast horizon (samples)', ylabel='Test RMSE (a.u.)')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# A fixed example index avoids cherry-picking a favourable prediction.
example = 0
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(np.arange(-15,1), X_test_raw[example,-16:,1], color='black', label='history y')
ax.plot(np.arange(1,H+1), Y_test_raw[example], 'ko-', linewidth=2, label='true future')
for name in ['Persistence','Ridge','Short / last','Dilated / last','Dilated / mean']:
    ax.plot(np.arange(1,H+1), test_predictions[name][example], '--', label=name)
ax.axvline(0, color='grey')
ax.set(xlabel='Samples relative to forecast origin', ylabel='Response (a.u.)')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 10. Exit ticket — 10 minutes

Submit the executed notebook plus approximately 250–400 words answering:

1. Why can a 64-sample input window still give a model only 17 samples of usable history at its final position?
2. Why is the receptive field 61 rather than 31 for our four dilated blocks?
3. What does the suffix-change experiment demonstrate? Can preprocessing still leak future information?
4. Why might final-position selection help this forecasting task? Why is global pooling nevertheless a valid candidate?
5. What changes between `Dense(10)` on `[B,L,C]` and on `[B,C]`?
6. Which model has the best error/simplicity trade-off in your run? Support the answer with horizon-specific results and identify limitations of the comparison.

**Optional challenge:** Repeat the neural comparison with three initialization seeds and report variation. Keep runs and data partitions fixed. Make choices on validation data. A separate new test set is needed if the original test results influenced subsequent redesign.

**Optional challenge:** Before regenerating data with another delay, predict the minimum final-position receptive field required to access the command affecting horizon one. Modify the delay and dilation schedule, then test the prediction. Keep $H\leq\mathrm{DELAY}$ for the supplied analytical reference.


## References and lecture alignment

The lab follows **05 CNNs(1).pptx**, the supplied Lecture 5 deck:

| Lecture slides | Notebook content |
|---|---|
| 99–124 | Receptive-field span, stride, accessible history |
| 125–144 | Causality and leakage checks |
| 145–168 | Dilation, coverage and temporal resolution |
| 169–193; 250–253 | Residual TCN and prediction heads |
| 194–212; 254–258 | Optional WaveNet-style gating and skip paths |
| 213–227; 245–249; 261–262 | Forecasting, pooling and output tensor shapes |
| 228–244; 259–260 | Optional parallel multiscale model |

Primary implementation references:
- [Keras Conv1D](https://keras.io/api/layers/convolution_layers/convolution1d/)
- [Keras GlobalAveragePooling1D](https://keras.io/api/layers/pooling_layers/global_average_pooling1d/)
- [Keras Dense](https://keras.io/api/layers/core_layers/dense/)
- Bai, Kolter and Koltun, [An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling](https://arxiv.org/abs/1803.01271), 2018.
- van den Oord and colleagues, [WaveNet: A Generative Model for Raw Audio](https://arxiv.org/abs/1609.03499), 2016.

The architectures here are small teaching implementations. They do not reproduce every detail or reported result of the referenced papers.
